# Price-point analysis

The BigQuery SQL returns analysis-ready subscription-level data. `summarize_price_points` groups that data by any requested dimensions and calculates user counts, contact rate, churn rate, save rate, gamer rate, revenue, and CNRC.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.examine_result_helpers import plot_metric, summarize_price_points
from src.refresh_price_point_raw_data import replace_csv

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Load detail data

In [2]:
project_id = "gannett-datascience"
sql_path = PROJECT_ROOT / "src" / "sql" / "price_point_detail_data.sql"
client = bigquery.Client(project=project_id)
df = client.query(sql_path.read_text()).to_dataframe(create_bqstorage_client=False)

print(
    f"Loaded {len(df):,} mature subscriptions across "
    f"{df['market'].nunique(dropna=True):,} markets."
)

Loaded 71,818 mature subscriptions across 210 markets.


## 2. Summaries

Pass any columns to `dimensions`. Subset `df` with normal pandas code when an analysis needs a specific population.

In [3]:
print("Overall summary:")
overall = summarize_price_points(df)
display(overall.round(2))

print("Summary by cohort and treatment:")
cohort_treatment = summarize_price_points(df, ["cohort", "treatment"])
display(cohort_treatment.round(2))

print("Summary by start price group and treatment of Three-Offer Cohort:")
three_offer = df.loc[df["cohort"].eq("Three-Offer Cohort")]
price_treatment = summarize_price_points(
    three_offer, ["start_price_group", "treatment"]
)
display(price_treatment.round(2))

print("Summary by gamer type and treatment of contacted users:")
contacted = df.loc[df["contacted"].eq(1)]
gamer_treatment = summarize_price_points(
    contacted, ["gamer_type", "treatment"]
)
display(
    gamer_treatment[[
        "gamer_type", "treatment", "total_users",
        "churn_rate", "total_revenue", "cnrc",
    ]].round(2)
)

Overall summary:


,overall,total_users,contacted_users,saved_users,stopped_users,gamer_users,total_revenue,contact_rate,churn_rate,save_rate,gamer_rate,cnrc
0,Overall,71818,14280,5405,8875,3185,"154,798.590",0.200,0.620,0.380,0.220,10.840


Summary by cohort and treatment:


,cohort,treatment,total_users,contacted_users,saved_users,stopped_users,gamer_users,total_revenue,contact_rate,churn_rate,save_rate,gamer_rate,cnrc
0,Three-Offer Cohort,Control,13494,2830,1639,1191,713,"41,466.360",0.210,0.420,0.580,0.250,14.650
1,Three-Offer Cohort,Midpoint,13427,2799,952,1847,690,"31,965.640",0.210,0.660,0.340,0.250,11.420
2,Three-Offer Cohort,Tiered,13355,2716,705,2011,683,"23,790.770",0.200,0.740,0.260,0.250,8.760
3,Two-Offer Cohort,Control,15848,3071,1374,1697,568,"33,827.470",0.190,0.550,0.450,0.180,11.020
4,Two-Offer Cohort,Midpoint,15694,2864,735,2129,531,"23,748.350",0.180,0.740,0.260,0.190,8.290


Summary by start price group and treatment of Three-Offer Cohort:


,start_price_group,treatment,total_users,contacted_users,saved_users,stopped_users,gamer_users,total_revenue,contact_rate,churn_rate,save_rate,gamer_rate,cnrc
0,1. <=4.99,Control,1326,634,512,122,184,"7,102.450",0.480,0.190,0.810,0.290,11.200
1,1. <=4.99,Midpoint,1325,608,290,318,178,"5,790.690",0.460,0.520,0.480,0.290,9.520
2,1. <=4.99,Tiered,1282,601,194,407,166,"4,410.650",0.470,0.680,0.320,0.280,7.340
3,2. <=9.99,Control,3508,1220,831,389,350,"23,691.590",0.350,0.320,0.680,0.290,19.420
4,2. <=9.99,Midpoint,3491,1235,425,810,328,"16,577.110",0.350,0.660,0.340,0.270,13.420
5,2. <=9.99,Tiered,3430,1148,344,804,330,"13,596.070",0.330,0.700,0.300,0.290,11.840
6,3. <=14.99,Control,3654,475,175,300,99,"5,894.760",0.130,0.630,0.370,0.210,12.410
7,3. <=14.99,Midpoint,3661,492,137,355,110,"5,067.900",0.130,0.720,0.280,0.220,10.300
8,3. <=14.99,Tiered,3683,468,90,378,104,"3,095.210",0.130,0.810,0.190,0.220,6.610
9,4. <=16.99,Control,12,0,0,0,0,0.000,0.000,NaN,NaN,NaN,NaN


Summary by gamer type and treatment of contacted users:


,gamer_type,treatment,total_users,churn_rate,total_revenue,cnrc
0,Not Gamer,Control,4620,0.580,"57,564.500",12.460
1,Not Gamer,Midpoint,4442,0.800,"33,336.830",7.500
2,Not Gamer,Tiered,2033,0.840,"11,093.660",5.460
3,Repeat Restart,Control,98,1.000,0.000,0.000
4,Repeat Restart,Midpoint,328,1.000,14.990,0.050
5,Repeat Restart,Tiered,235,1.000,15.990,0.070
6,Seek and paid lower rate,Control,31,0.000,"1,380.830",44.540
7,Seek and paid lower rate,Midpoint,204,0.010,"9,949.840",48.770
8,Seek and paid lower rate,Tiered,114,0.010,"6,611.490",58.000
9,"Seek lower rate, still paid offered rate",Control,1152,0.100,"16,348.500",14.190


In [4]:
print("Summary by market with at least 200 total users (descending ordered by contact rate):")
market_summary = (
    summarize_price_points(df, ["market"])
    .loc[lambda result: result["total_users"].ge(200)]
    .sort_values("contact_rate", ascending=False)
)
display(market_summary.head(30).round(2))

market_summary_path = PROJECT_ROOT / "src/notebooks/tables/examine_result/market_summary.csv"
replace_csv(market_summary_path, market_summary)
print(f"Saved market summary to {market_summary_path}")

Summary by market with at least 200 total users (descending ordered by contact rate):


,market,total_users,contacted_users,saved_users,stopped_users,gamer_users,total_revenue,contact_rate,churn_rate,save_rate,gamer_rate,cnrc
175,SpringfieldMO,336,124,47,77,19,"1,415.340",0.370,0.620,0.380,0.150,11.410
9,Appleton,274,101,57,44,33,"1,981.250",0.370,0.440,0.560,0.330,19.620
170,Sioux Falls,226,81,33,48,19,"1,209.300",0.360,0.590,0.410,0.230,14.930
46,Detroit,2317,813,378,435,218,"11,035.420",0.350,0.540,0.460,0.270,13.570
84,Indianapolis,1020,346,137,209,89,"4,414.870",0.340,0.600,0.400,0.260,12.760
45,Des Moines,757,255,108,147,65,"3,525.830",0.340,0.580,0.420,0.250,13.830
142,Peoria,315,106,52,54,32,"1,915.370",0.340,0.510,0.490,0.300,18.070
58,Fort Collins,501,165,72,93,35,"2,233.520",0.330,0.560,0.440,0.210,13.540
98,Lansing,449,139,64,75,29,"1,985.640",0.310,0.540,0.460,0.210,14.290
22,Brevard,348,106,43,63,26,"1,065.920",0.300,0.590,0.410,0.250,10.060


Saved market summary to /Users/ying/Documents/DEV/work/stop_save_pChurn/src/notebooks/tables/examine_result/market_summary.csv


## 3. Charts

In [ ]:
plot_metric(
    cohort_treatment,
    metric="contact_rate",
    x="cohort",
    hue="treatment",
    title="Contact rate by cohort and treatment",
)
plt.show()

plot_metric(
    price_treatment,
    metric="churn_rate",
    x="start_price_group",
    hue="treatment",
    title="Three-Offer churn rate by start-price group and treatment",
)
plt.show()

plot_metric(
    gamer_treatment,
    metric="cnrc",
    x="gamer_type",
    hue="treatment",
    title="CNRC by gamer type and treatment",
)
plt.show()